# MODIS NDVI — Porto Alegre

Pipeline: GEE ImageCollection (MOD13Q1) → annual mean composite → COG → visual + value tiles.

**Source:** `MODIS/061/MOD13Q1` (NDVI band, scale 0.0001)  
**Outputs:** `output/ndvi_cog.tif`, `output/tiles_visual/`, `output/tiles_values/`, `output/ndvi_metadata.json`

**Requires:** `earthengine-api`, GDAL

## Step 1. GEE — Export annual mean NDVI composite

In [ ]:
import ee

ee.Initialize(project='citycatalyst')

In [ ]:
# Porto Alegre bounding box [minLon, minLat, maxLon, maxLat]
poa = ee.Geometry.Rectangle([-51.40, -30.42, -50.82, -29.88])

# MOD13Q1 NDVI — annual mean composite
# Raw NDVI scale: 0.0001 (values -2000 to 10000 → -0.2 to 1.0)
year = "2024"
ndvi_raw = (
    ee.ImageCollection("MODIS/061/MOD13Q1")
    .filterBounds(poa)
    .filterDate(f"{year}-01-01", f"{int(year)+1}-01-01")
    .select("NDVI")
    .mean()
)

# Scale to -1..1 (multiply raw by 0.0001)
ndvi_scaled = ndvi_raw.multiply(0.0001)

# Clip and reproject to Web Mercator (250m native)
img = ndvi_scaled.clip(poa).reproject(crs="EPSG:3857", scale=250)

task = ee.batch.Export.image.toDrive(
    image=img,
    description=f"poa_modis_ndvi_{year}",
    folder="gee_exports",
    fileNamePrefix=f"poa_modis_ndvi_{year}",
    region=poa,
    scale=250,
    crs="EPSG:3857",
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

task.start()
print("Started Drive export:", task.id)
print("Download the GeoTIFF from Drive to data/poa_modis_ndvi_2024.tif before running Step 2.")

In [ ]:
import time

def wait_for_task(task):
    """Poll task status until complete or failed."""
    while task.active():
        status = task.status()
        state = status.get("state", "UNKNOWN")
        print(f"  Status: {state}")
        time.sleep(10)
    status = task.status()
    if status.get("state") == "COMPLETED":
        print("Export finished successfully.")
    else:
        print("Export ended with status:", status)

# wait_for_task(task)

## Step 2. GDAL — COG and tiles

Place the downloaded GeoTIFF in `data/poa_modis_ndvi_2024.tif`, then run the cells below.

In [ ]:
import subprocess
import json
from pathlib import Path

# Paths (run from release/v1/2024/)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
IN_TIF = DATA_DIR / "poa_modis_ndvi_2024.tif"
OUT_COG = OUTPUT_DIR / "ndvi_cog.tif"
OUT_COLORIZED = OUTPUT_DIR / "ndvi_colorized.tif"
OUT_VALUE_ENCODED = OUTPUT_DIR / "ndvi_value_encoded.tif"
TILES_VISUAL = OUTPUT_DIR / "tiles_visual"
TILES_VALUES = OUTPUT_DIR / "tiles_values"
COLORS_VISUAL = DATA_DIR / "ndvi_visual_colors.txt"
COLORS_VALUE = DATA_DIR / "ndvi_value_encoding_colors.txt"
METADATA_JSON = OUTPUT_DIR / "ndvi_metadata.json"

# Terrain RGB encoding params (value = (R*256² + G*256 + B) * scale + offset)
NDVI_SCALE = 0.001
NDVI_OFFSET = -1.0
NDVI_UNIT = ""

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not IN_TIF.exists():
    raise FileNotFoundError(f"Download GeoTIFF from Drive first. Expected: {IN_TIF}")

In [ ]:
# Already in EPSG:3857 from GEE; convert to COG
subprocess.run([
    "gdal_translate", str(IN_TIF), str(OUT_COG),
    "-of", "COG",
    "-co", "COMPRESS=DEFLATE",
    "-co", "OVERVIEWS=AUTO",
    "-co", "RESAMPLING=AVERAGE"
], check=True)
print("Created COG:", OUT_COG)

In [ ]:
# Colorize for visual tiles
subprocess.run([
    "gdaldem", "color-relief", str(OUT_COG), str(COLORS_VISUAL), str(OUT_COLORIZED)
], check=True)
print("Colorized for display:", OUT_COLORIZED)

In [ ]:
# Generate visual XYZ tiles
TILES_VISUAL.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(OUT_COLORIZED), str(TILES_VISUAL)
], check=True)
print("Visual tiles written to:", TILES_VISUAL)

In [ ]:
# Value tiles (Terrain RGB) for hover lookup
subprocess.run([
    "gdaldem", "color-relief", str(OUT_COG), str(COLORS_VALUE), str(OUT_VALUE_ENCODED)
], check=True)

TILES_VALUES.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(OUT_VALUE_ENCODED), str(TILES_VALUES)
], check=True)
print("Value tiles written to:", TILES_VALUES)

In [ ]:
# Write metadata for frontend
metadata = {
    "layer": "ndvi",
    "description": "Normalized Difference Vegetation Index (-1 to 1)",
    "time_period": "2024",
    "encoding": {
        "type": "terrain_rgb",
        "scale": NDVI_SCALE,
        "offset": NDVI_OFFSET,
        "unit": NDVI_UNIT,
        "decode_formula": "value = (R * 256 * 256 + G * 256 + B) * scale + offset"
    }
}
with open(METADATA_JSON, "w") as f:
    json.dump(metadata, f, indent=2)
print("Metadata written to:", METADATA_JSON)